# PO-034: finite product residuals and higher-cell evidence

Technical preprint v0.1 (0.1.0). Authority effect: None.

Install the package with the pinned code/uv.lock before execution.
This notebook uses NumPy floating-point arithmetic. It distinguishes Q from
R_k, compares an analytic state with a proved bound, and keeps scalar Hodge
calculations separate from quantum interpretation.

In [1]:
import numpy as np
from coda_higher_cell import (
    k_cell_tangent_projector, k_cell_normal_projector, highest_order_correlation_projector,
    verify_marginal_preservation, verify_generalized_tangent_isolation,
    SimplicialComplex, simplicial_boundary_matrix, simplicial_laplacian, hodge_decomposition,
    short_time_k_party_defect_lower_bound, schatten1_dual_witness,
    PrimeTriadCarrier, prime_triad_carrier_declaration, can_continue_predicate,
)
from coda_higher_cell.k_cell_retraction import tensor_product_list, embed_local_operator
from coda_higher_cell.defect_bounds import schatten_1_norm, evaluate_k_party_trace_defect
from coda_higher_cell.bicomplex_ddc import DoubleBicomplex, bicomplex_curvature_tensor, ddc_trace_sequence_evaluation
from coda_higher_cell.prime_triad_bridge import adapter_loss_ledger
print("Finite numerical modules imported.")

Finite numerical modules imported.


## 1. Product projector: idempotence and single-site marginals

Three subsystems form a degree-2 simplex. The numerical checks below use mixed, full-rank reference factors.

In [2]:
dims = [2, 2, 2]
rng = np.random.default_rng(42)
marginals = []
for d in dims:
    a = rng.normal(size=(d,d)) + 1j*rng.normal(size=(d,d))
    rho = a @ a.conj().T
    marginals.append(rho/np.trace(rho))
a = rng.normal(size=(8,8)) + 1j*rng.normal(size=(8,8))
z = a + a.conj().T
z -= np.trace(z)*np.eye(8)/8
pz = k_cell_tangent_projector(z,marginals,dims)
error = np.linalg.norm(k_cell_tangent_projector(pz,marginals,dims)-pz)
assert error < 1e-11 and verify_marginal_preservation(z,marginals,dims)
print(f"Idempotence error: {error:.3e}; single-site marginals preserved.")

Idempotence error: 1.973e-15; single-site marginals preserved.


## 2. A two-body counterexample inside a triangle

One-site terms are tangent. For \(H=X_1X_2I_3\) at \(|000\rangle\),
the product-state residual QL is nonzero, while the distinct highest-order
component R3 L vanishes.

In [3]:
x = np.array([[0,1],[1,0]],dtype=complex)
rho0 = np.diag([1.,0.])
reference = [rho0]*3
rho_product = tensor_product_list(reference)
h_two = embed_local_operator(np.kron(x,x),[0,1],dims)
l_two = -1j*(h_two @ rho_product-rho_product @ h_two)
q_two = k_cell_normal_projector(l_two,reference,dims)
r_two = highest_order_correlation_projector(l_two,reference,dims)
assert verify_generalized_tangent_isolation(x,[0],reference,dims)
assert not verify_generalized_tangent_isolation(np.kron(x,x),[0,1],reference,dims)
assert np.allclose(q_two,l_two) and np.allclose(r_two,0)
assert abs(schatten_1_norm(q_two)-2) < 1e-12
print(f"Two-body Q residual norm: {schatten_1_norm(q_two):.6f}; R3 norm: {schatten_1_norm(r_two):.3e}")

Two-body Q residual norm: 2.000000; R3 norm: 0.000e+00


## 3. Actual GHZ-family states versus the Hamiltonian bound

Here \(\|H\|_\infty=g=1\) for the entire closed-system Hamiltonian.
Construct the analytic state, not just the bound polynomial.
The distance is to the product of marginals; it is not a separability test.

In [4]:
h_ghz = tensor_product_list([x]*3)
l_ghz = -1j*(h_ghz @ rho_product-rho_product @ h_ghz)
nu = schatten_1_norm(k_cell_normal_projector(l_ghz,reference,dims))
w_ghz,norm_ghz,err_ghz = schatten1_dual_witness(l_ghz)
assert abs(nu-2) < 1e-12 and err_ghz < 1e-12
for t in [0.,0.001,0.005,1/54]:
    psi = np.zeros(8,dtype=complex)
    psi[0],psi[7] = np.cos(t),-1j*np.sin(t)
    defect = evaluate_k_party_trace_defect(np.outer(psi,psi.conj()),dims)
    bound,threshold,in_window = short_time_k_party_defect_lower_bound(nu,3,1,t)
    assert in_window and defect+1e-12 >= bound
    print(f"t={t:.6f}: numerical defect={defect:.8f}, lower bound={bound:.8f}")
fixed_b = np.zeros((8,8),dtype=complex)
fixed_b[7,0],fixed_b[0,7] = -1j,1j
observed_scalar = float(np.vdot(fixed_b,l_ghz).real)
print(f"Fixed linear trace observation on the degree-2 cell: {observed_scalar:.6f}")

t=0.000000: numerical defect=0.00000000, lower bound=0.00000000
t=0.001000: numerical defect=0.00200300, lower bound=0.00194600
t=0.005000: numerical defect=0.01007486, lower bound=0.00865000
t=0.018519: numerical defect=0.03805849, lower bound=0.01851852
Fixed linear trace observation on the degree-2 cell: 2.000000


## 4. Polar optimality does not imply tangent annihilation

A diagonal normal residual at a mixed product reference has a norm-optimal polar witness with nonzero pairing against a tangent vector.

In [5]:
rhoa,rhob = np.diag([0.75,0.25]),np.diag([1/3,2/3])
n = np.diag([1,-1,-1,1])
w,norm_w,alignment = schatten1_dual_witness(n)
tangent = np.kron(np.diag([1,-1]),rhob)
assert np.allclose(k_cell_normal_projector(n,[rhoa,rhob],[2,2]),n)
assert norm_w <= 1+1e-12 and alignment < 1e-12
pairing = np.vdot(w,tangent).real
assert abs(pairing+2/3) < 1e-12
classical = np.diag([0.5,0,0,0.5])
assert abs(evaluate_k_party_trace_defect(classical,[2,2])-1) < 1e-12
print(f"Optimal norm witness; tangent pairing = {pairing:.6f}. Separable classical defect = 1.")

Optimal norm witness; tangent pairing = -0.666667. Separable classical defect = 1.


## 5. Ordinary scalar Hodge decomposition

Edge order is (01),(02),(12). A hollow triangle has harmonic vector (1,-1,1); a filled triangle has no degree-1 harmonic mode. No protected-entanglement interpretation is asserted.

In [6]:
hollow = SimplicialComplex([(0,1),(0,2),(1,2)])
filled = SimplicialComplex([(0,1,2)])
cycle = np.array([1.,-1.,1.])
exact,coexact,harmonic = hodge_decomposition(cycle,hollow,1)
assert np.allclose(harmonic,cycle)
b1,b2 = simplicial_boundary_matrix(filled,1),simplicial_boundary_matrix(filled,2)
assert np.array_equal(b1 @ b2,np.zeros((3,1)))
assert np.array_equal(simplicial_laplacian(filled,1),3*np.eye(3))
print("Hollow harmonic component:",harmonic.real)
print("Filled triangle Delta1 eigenvalues:",np.linalg.eigvalsh(simplicial_laplacian(filled,1)))

Hollow harmonic component: [ 1. -1.  1.]
Filled triangle Delta1 eigenvalues: [3. 3. 3.]


## 6. A typed square and a truthful seven-label schema

C=dh dv-dv dh; the mixed D^2 block is (-1)^p C. The demonstration double complex is nilpotent and commuting. The protocol helper does not execute or verify quantum operations.

In [7]:
dh = np.array([[1.,-1.]])
square = bicomplex_curvature_tensor(dh,dh,np.eye(2),np.array([[2.]]))
assert np.array_equal(square,-dh)
double = DoubleBicomplex(filled)
assert np.allclose(double.vertical_differential(0,1) @ double.vertical_differential(0,0),0)
receipt = ddc_trace_sequence_evaluation("triangle",nu)
assert len(receipt["trace_sequence"]) == 7 and not receipt["receipt_verified"]
loss = adapter_loss_ledger(np.diag([1,-1]),[-np.diag([1,-1])])
assert abs(loss-4) < 1e-12
carrier = PrimeTriadCarrier(
    "triangle",2,(0,1,2),nu,loss,bool(norm_ghz <= 1+1e-12 and err_ghz < 1e-12),80,
    "notebook:triangle-incidence","notebook:fixed-trace-observation","notebook:GHZ-SVD","state:GHZ-initial",
)
declaration = prime_triad_carrier_declaration(carrier)
allowed,reason = can_continue_predicate(carrier,3)
assert declaration["declaration_valid"] and allowed
print("Square defect:",square.real)
print("Schema labels:",receipt["trace_sequence"],"; verified:",receipt["receipt_verified"])
print("Carrier check scope:",declaration["validation_scope"])
print("Continuation policy:",reason)

Square defect: [[-1.  1.]]
Schema labels: ['E', 'R', 'C', 'Lambda', 'D', 'M', 'T'] ; verified: False
Carrier check scope: metadata completeness; evidence contents not replayed
Continuation policy: CONTINUE_ASCENT: local policy permits a next check; no authority is granted
